# Lab 5 · Saving, Merging & Model Formats

**~20 minutes.**

A trained adapter in a dying Kaggle session is worth nothing. This
notebook covers getting it out, in the right format for the right runtime.

⚠️ **Disk warning.** `/kaggle/working` is ~20 GB. A merged 16-bit model is
6.4 GB and GGUF conversion needs room for both. We export **one**
quantization and clean up as we go. Watch the free-space numbers.

> ↳ Slides: *Save Model* · *Model Format: GGUF, GPTQ, AWQ, ONNX, TensorRT, EXL2*

In [ ]:
# --- Install the fine-tuning stack --------------------------------------
# Kaggle ships a matched torch/CUDA pair. --no-deps stops pip replacing torch
# with an incompatible build, which shows up later as baffling CUDA errors.
#
# Takes 2-4 minutes. "dependency conflict" warnings here are expected and fine.
# Output is deliberately NOT suppressed: if this step fails, you need to see it.
!pip install -q --no-deps unsloth unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets huggingface_hub sentencepiece protobuf

print("\ninstall finished - verifying imports...")
import importlib
missing = [m for m in ("unsloth", "trl", "peft", "bitsandbytes", "datasets")
           if importlib.util.find_spec(m) is None]
print("MISSING: " + ", ".join(missing) if missing else "all packages importable")

In [ ]:
# --- Locate the workshop repo -------------------------------------------
# Tries, in order: already present -> attached Kaggle Dataset -> git clone.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/YOUR-USERNAME/LLM-lab.git"   # TODO: your repo

def find_repo() -> Path:
    for candidate in [Path("/kaggle/working/LLM-lab"), Path.cwd(), Path.cwd().parent]:
        if (candidate / "common" / "config.py").exists():
            return candidate
    for d in Path("/kaggle/input").glob("*"):          # attached as a Dataset
        if (d / "common" / "config.py").exists():
            return d
    print("Repo not found locally, cloning...")        # last resort
    subprocess.run(["git", "clone", "-q", REPO_URL, "/kaggle/working/LLM-lab"], check=True)
    return Path("/kaggle/working/LLM-lab")

REPO = find_repo()
sys.path[:0] = [str(REPO / "common"), str(REPO / "dataset")]
print(f"repo: {REPO}")

import config
print(config.summary())

In [ ]:
# --- Hugging Face authentication ----------------------------------------
# Reads the Kaggle Secret named HF_TOKEN. Never paste a token into a cell:
# it is saved with the notebook and shared notebooks leak tokens constantly.
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("Hugging Face: authenticated")
except Exception as e:
    print(f"Hugging Face: NOT authenticated ({type(e).__name__})")
    print("  Fix: right panel -> Add-ons -> Secrets -> add HF_TOKEN, tick the box.")
    print("  Or set USE_UNGATED_MODEL = True below to skip the gated model entirely.")

In [ ]:
import shutil, subprocess

def disk(label=""):
    free = shutil.disk_usage("/kaggle/working")[2] / 2**30
    print(f"  {label:<34} {free:5.1f} GB free")
    return free

def size_of(path):
    out = subprocess.run(["du", "-sh", str(path)], capture_output=True, text=True)
    return out.stdout.split()[0] if out.returncode == 0 else "?"

disk("starting point")

## 6.1 Three ways to save

| mode | size | what it is | use when |
|---|---|---|---|
| **adapter only** | ~90 MB | just the LoRA matrices | sharing, versioning, serving many adapters |
| **merged 16-bit** | ~6.4 GB | ΔW folded into W | vLLM, further conversion, HF Hub |
| **merged 4-bit** | ~2.2 GB | merged then requantized | quick deploy, some quality loss |

The adapter is already saved from Lab 3.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = str(config.ADAPTER_DIR),
    max_seq_length = config.MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = config.LOAD_IN_4BIT,
)
print(f"adapter directory: {size_of(config.ADAPTER_DIR)}")
for f in sorted(config.ADAPTER_DIR.iterdir()):
    if f.stat().st_size > 1024:
        print(f"  {f.name:<36} {f.stat().st_size/2**20:>8.1f} MB")

## 6.2 Why you'd keep the adapter separate

90 MB versus 6.4 GB is a 70× difference, and it changes how you deploy.

Forty fine-tuned variants as merged models is 256 GB and forty things to
load. Forty adapters is 3.6 GB on top of one base model — and vLLM can
serve them from a single loaded copy, switching per request (Lab 7).

Merge when a runtime demands a single artifact — GGUF conversion does.

## 6.4 Merge to 16-bit, then convert to GGUF

**GGUF** is llama.cpp's format: one self-contained file with weights,
tokenizer and metadata, designed for CPU/GPU inference on consumer
hardware. It's what Ollama consumes.

`q4_k_m` — "4-bit, K-quant, medium" — is the standard quality/size
compromise. Roughly 2 GB for a 3B model, with quality loss most people
can't detect in chat.

**This takes 5–10 minutes** and is the most disk-hungry step of the day.

In [ ]:
disk("before GGUF export")

# Unsloth merges to 16-bit internally, then calls llama.cpp's converter.
model.save_pretrained_gguf(
    str(config.GGUF_DIR),
    tokenizer,
    quantization_method = config.GGUF_QUANT,
)

disk("after GGUF export")

In [ ]:
gguf_files = sorted(config.GGUF_DIR.glob("*.gguf"))
print("GGUF files produced:")
for f in gguf_files:
    print(f"  {f.name:<44} {f.stat().st_size/2**30:>6.2f} GB")

if gguf_files:
    GGUF_PATH = gguf_files[0]
    print(f"\nLab 6 will use: {GGUF_PATH}")
else:
    print("\nNo .gguf produced - see docs/troubleshooting.md")

### What the other quantizations would have cost

We only built one because of the disk limit. For reference, same model:

| quant | size | quality |
|---|---|---|
| `f16` | ~6.4 GB | lossless vs the merged model |
| `q8_0` | ~3.4 GB | essentially indistinguishable |
| `q5_k_m` | ~2.3 GB | very good |
| **`q4_k_m`** | **~2.0 GB** | **good — the usual default** |
| `q3_k_m` | ~1.6 GB | noticeably degraded |
| `q2_k` | ~1.3 GB | often incoherent |

The curve is steep at the bottom: below 4-bit, quality falls off fast for
a small size win. `q4_k_m` is where most people land.

## 6.3 Push to the Hub — get it off Kaggle

Kaggle sessions expire. Whatever isn't pushed or downloaded is gone.

Pushing the **adapter** is the cheap option: 90 MB, and anyone can
reconstruct the full model from it.

Set `LLMLAB_HUB_ID` in `common/config.py`, or edit below.

In [ ]:
HUB_ID = config.HUB_MODEL_ID       # e.g. "yourname/aura-support-3b-lora"

if HUB_ID:
    model.push_to_hub(HUB_ID, token=os.environ.get("HF_TOKEN"))
    tokenizer.push_to_hub(HUB_ID, token=os.environ.get("HF_TOKEN"))
    print(f"pushed -> https://huggingface.co/{HUB_ID}")
else:
    print("HUB_ID not set - skipping.")
    print("Set config.HUB_MODEL_ID (or LLMLAB_HUB_ID) to push.")
    print("\nAlternative: download the GGUF from the Kaggle output panel")
    print("on the right, under /kaggle/working.")

## 6.6 Which format goes with which runtime

The question that trips people up: *why can't I just use my model
anywhere?* Because each runtime wants its weights laid out its own way.

| format | runtime | precision | notes |
|---|---|---|---|
| **safetensors** | HF transformers, vLLM | fp16/bf16/4-bit | the interchange default |
| **GGUF** | llama.cpp, Ollama, LM Studio | 2–8 bit K-quants | single file, CPU-friendly |
| **GPTQ** | vLLM, AutoGPTQ | 3/4/8-bit | needs calibration data |
| **AWQ** | vLLM, AutoAWQ | 4-bit | activation-aware; often beats GPTQ |
| **EXL2** | ExLlamaV2 | variable | fast on consumer NVIDIA |
| **ONNX** | ONNX Runtime | various | cross-platform, incl. mobile |
| **TensorRT-LLM** | Triton | various | fastest on NVIDIA, most work to build |

The two rules worth memorising:

- **You cannot feed GGUF to vLLM** — vLLM wants safetensors (or GPTQ/AWQ). GGUF is llama.cpp's.
- **You cannot feed safetensors to Ollama** — Ollama is llama.cpp underneath and wants GGUF.

Which is why Lab 6 uses the GGUF and Lab 7 uses the adapter directly.

In [ ]:
disk("end of Lab 5")
print("\n  Keep for later:")
print(f"    {config.ADAPTER_DIR}   (Lab 7, vLLM)")
print(f"    {config.GGUF_DIR}      (Lab 6, Ollama)")

---

### Next: `06_serve_ollama.ipynb` — serve it with Ollama

> **Kaggle tip:** if the session has been idle a while, check the right-hand
> panel still shows the GPU attached before starting the next notebook.